# Bengali NLP Safety — V3: corrected training and controlled tuning

**Your reported Kaggle baseline: 0.516. No new score has been measured yet.**
This revision fixes code problems and runs real experiments when you execute it with your competition data and a GPU.

## Start here
1. Open this notebook in **Google Colab**. Select **Runtime → Change runtime type → T4 GPU** (or another GPU).
2. Run the installation cell first. If Colab asks for a restart, restart before continuing.
3. Leave `PRESET = "standard"` for the first run. It trains two BanglaBERT configurations sequentially; this can take hours depending on data size and GPU. `quick` trains one; `thorough` trains five.
4. Run the remaining cells in order. Upload `kaggle.json` if prompted. Accept the competition rules on Kaggle before downloading.
5. Download `submission_optimized_v3.csv` and submit **that CSV**, not the notebook or probability files.
6. Keep `experiment_report.json`, `per_label_audit.csv`, `experiment_leaderboard.csv`, and the prediction archive for the next review.

**Assumptions to check:** the supplied notebook assumes seven binary targets, `ID` / `text` columns, and macro F1. I could not independently retrieve the competition evaluation/rules page. Verify the metric, binary submission format, and permission to use these pretrained models on the competition page. No external labeled dataset is added. If the metric differs, this tuning objective must change.

## What was wrong / what changed
| Original notebook | V3 |
|---|---|
| Multiple seeds were trained but never averaged | Explicit probability averages, with identical members at validation and test time |
| Selected primary seed could be replaced by seed 42 at inference | Every run saves its own validation and test predictions together |
| Only whitespace cleaning for BanglaBERT | Official BUET normalizer for BanglaBERT/BanglishBERT; conservative text for TF-IDF/XLM-R |
| Learning rate fixed at 2e-5 | Automatic 2e-5 versus 1e-5 experiment in standard mode |
| Same validation labels repeatedly used for every choice | Separate development, calibration, and final audit groups |
| TF-IDF refitted after calibration, changing its probabilities | Exactly the calibrated fitted models generate the submission |
| Best checkpoint copied on GPU; models retained on GPU | Best weights on disk; sequential training/inference and GPU cleanup |
| Torch RNG not seeded; final accumulation group under-scaled | Torch seeds and sample-correct gradient accumulation |
| Fixed text length 128 with no diagnostics | Default 192; truncation rates printed; 256 available |
| Test row order only checked against test.csv | Submission reordered and validated against sample_submission.csv |

Fixes are real; score improvement is an experiment, not a guarantee. The uploaded notebook contained no execution outputs, so its actual validation scores, chosen weights, and training history cannot be recovered.

In [ ]:
# 1. Install before importing libraries. Keep Colab's preinstalled GPU-enabled PyTorch.
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.2", "tokenizers==0.19.1", "sentencepiece==0.2.0",
    "kaggle", "scikit-learn", "pandas", "scipy", "tqdm", "safetensors",
    "git+https://github.com/csebuetnlp/normalizer.git"])
print("Dependencies installed. If prompted to restart, do so before running the next cell.")

In [ ]:
# 2. Settings — edit only this cell for your next experiment.
from pathlib import Path
PRESET = "standard"                 # quick: 1 GPU fit; standard: 2; thorough: 5
DATA_DIR = Path("/content/data")
USE_DRIVE = True                     # recommended: preserves completed runs after Colab disconnects
DRIVE_OUTPUT = "/content/drive/MyDrive/bengali_safety_v3"
AUTO_DOWNLOAD_DATA = True
MAX_LENGTH = 192                     # try 256 later if truncation is substantial
TRAIN_BATCH_SIZE = 8                 # GPU out of memory? change to 4
ACCUMULATION_STEPS = 2               # if batch becomes 4, use 4 here to keep effective batch size 16
EVAL_BATCH_SIZE = 16
MAX_EPOCHS = 5
PATIENCE = 2
SEED = 42
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
REUSE_COMPLETED_RUNS = True
LABELS = ["is_toxic", "implicit_toxicity", "sarcasm", "contains_profanity",
          "threatening", "identity_attack", "sexual_content"]
ID_COL, TEXT_COL = "ID", "text"
# Macro F1 + binary predictions are inherited assumptions, not independently verified rules.
METRIC = "macro_f1"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUTPUT_ROOT = Path(DRIVE_OUTPUT if USE_DRIVE else "/content/bengali_safety_v3")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert PRESET in {"quick", "standard", "thorough"}
assert TRAIN_BATCH_SIZE > 0 and ACCUMULATION_STEPS > 0
assert METRIC == "macro_f1", "This notebook's selection code implements macro F1 only."
print("Preset:", PRESET, "| effective training batch:", TRAIN_BATCH_SIZE * ACCUMULATION_STEPS)

In [ ]:
# 3. Get competition data. Existing files are reused; your API key is never printed or saved in reports.
import os, zipfile, shutil, subprocess
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.rglob("train.csv")) and AUTO_DOWNLOAD_DATA:
    key = Path.home() / ".kaggle" / "kaggle.json"
    if not key.exists() and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) and not os.environ.get("KAGGLE_API_TOKEN"):
        from google.colab import files
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise ValueError("Upload kaggle.json, or put the competition CSV files in DATA_DIR.")
        key.parent.mkdir(parents=True, exist_ok=True)
        key.write_bytes(uploaded["kaggle.json"])
        key.chmod(0o600)
        Path("kaggle.json").unlink(missing_ok=True)
        del uploaded
    subprocess.run(["kaggle", "competitions", "download", "-c",
                    "bengali-nlp-safety-challenge-2026", "-p", str(DATA_DIR)], check=True)
    archive = DATA_DIR / "bengali-nlp-safety-challenge-2026.zip"
    if not archive.exists():
        raise FileNotFoundError("Expected competition ZIP missing. Inspect the download output.")
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            dest = (DATA_DIR / member.filename).resolve()
            if not dest.is_relative_to(DATA_DIR.resolve()):
                raise ValueError("Unsafe ZIP path")
        z.extractall(DATA_DIR)
roots = [p.parent for p in DATA_DIR.rglob("train.csv") if (p.parent / "test.csv").exists()]
if len(roots) != 1:
    raise ValueError(f"Expected one dataset folder; found {roots}. Set DATA_DIR to the correct folder.")
DATA_DIR = roots[0]
print("Data folder:", DATA_DIR)

In [ ]:
# 4. Imports, reproducibility, and input checks.
import gc, re, json, math, random, hashlib, unicodedata, warnings, importlib.metadata
import numpy as np
import pandas as pd
import torch
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from normalizer import normalize as buet_normalize
from tqdm.auto import tqdm
from IPython.display import display

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab: Runtime → Change runtime type → T4 GPU, then rerun.")
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("Exact repeatability can still vary across GPU types / software versions.")
train_df = pd.read_csv(DATA_DIR / "train.csv", dtype={ID_COL: str})
valid_df = pd.read_csv(DATA_DIR / "validation.csv", dtype={ID_COL: str})
test_df = pd.read_csv(DATA_DIR / "test.csv", dtype={ID_COL: str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={ID_COL: str})
for name, df, labeled in [("train", train_df, True), ("validation", valid_df, True), ("test", test_df, False)]:
    needed = [ID_COL, TEXT_COL] + (LABELS if labeled else [])
    assert set(needed).issubset(df.columns), f"{name}: missing columns {set(needed)-set(df.columns)}"
    assert len(df) > 0 and df[ID_COL].notna().all() and df[ID_COL].is_unique, f"{name}: invalid IDs"
    if labeled:
        for label in LABELS:
            df[label] = pd.to_numeric(df[label], errors="raise")
        assert df[LABELS].isin([0, 1]).all().all(), f"{name}: labels must be complete binary 0/1"
    print(name, df.shape, "| missing text:", df[TEXT_COL].isna().sum())
assert set(sample.columns) == {ID_COL, *LABELS}, "Sample submission schema differs; review the competition format."
assert sample[ID_COL].is_unique and sample[ID_COL].notna().all()
assert len(sample) == len(test_df) and set(sample[ID_COL]) == set(test_df[ID_COL])

## Validation in simple English
**Training** teaches the model. **Development** chooses the checkpoint and TF-IDF settings. **Calibration** chooses the model blend and decision thresholds. **Audit** is checked once, after all choices are frozen.

The supplied validation file is divided about 40% / 30% / 30% by text groups. The group split searches for similar label proportions; it is an approximate balancing heuristic, not a guarantee. Very rare labels may still be absent or unreliable. The audit remains useful only if you do not repeatedly change settings in response to its score. No validation labels enter model training in this version.

In [ ]:
# 5. Model-specific normalization and duplicate-aware validation.
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(value))).strip()

def bert_text(value):
    value = clean_text(value)
    return buet_normalize(value) if value else ""

for df in [train_df, valid_df, test_df]:
    df["clean_text"] = df[TEXT_COL].map(clean_text)
    df["bert_text"] = df["clean_text"].map(bert_text)
    # Group using the more aggressively normalized view to catch preprocessing collisions.
    df["group"] = df["bert_text"].map(clean_text)

# Keep validation untouched; remove train rows sharing normalized text or ID with validation.
leak = train_df["group"].isin(valid_df["group"]) | train_df[ID_COL].isin(valid_df[ID_COL])
print("Train rows excluded for validation overlap:", int(leak.sum()))
train_df = train_df.loc[~leak].reset_index(drop=True)
conflicts = train_df.groupby("group")[LABELS].nunique().gt(1).any(axis=1)
print("Train text groups with conflicting annotations:", int(conflicts.sum()), "(kept; do not invent labels)")
before = len(train_df)
train_df = train_df.drop_duplicates(["group"] + LABELS).reset_index(drop=True)
print("Exact duplicate text+label training rows removed:", before-len(train_df))
assert len(train_df) > 1, "Too few training rows remain after overlap checks."
print("Train/test normalized text overlap groups (reported only):", len(set(train_df.group) & set(test_df.group)))

def balanced_group_split(df, test_size, seed):
    if df["group"].nunique() < 4:
        raise ValueError("Too few independent text groups for reliable validation splitting.")
    y = df[LABELS].to_numpy(dtype=int)
    splitter = GroupShuffleSplit(n_splits=64, test_size=test_size, random_state=seed)
    best = None
    for left, right in splitter.split(df, y, groups=df["group"]):
        # Penalize missing classes strongly, then match positive proportions and row fraction.
        missing = sum(int(((y[idx].sum(0)==0) | (y[idx].sum(0)==len(idx))).sum()) for idx in [left,right])
        imbalance = np.abs(y[left].mean(0)-y[right].mean(0)).mean()
        cost = missing * 10 + imbalance + abs(len(right)/len(df)-test_size)
        if best is None or cost < best[0]:
            best = (cost, left, right)
    return best[1], best[2]

dev_idx, remainder = balanced_group_split(valid_df, .60, SEED)
cal_local, audit_local = balanced_group_split(valid_df.iloc[remainder], .50, SEED+1)
cal_idx, audit_idx = remainder[cal_local], remainder[audit_local]
assert set(dev_idx).isdisjoint(cal_idx) and set(dev_idx).isdisjoint(audit_idx) and set(cal_idx).isdisjoint(audit_idx)
assert len(dev_idx)+len(cal_idx)+len(audit_idx) == len(valid_df)
for a,b in [(dev_idx,cal_idx),(dev_idx,audit_idx),(cal_idx,audit_idx)]:
    assert set(valid_df.iloc[a].group).isdisjoint(valid_df.iloc[b].group)
y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_valid = valid_df[LABELS].to_numpy(dtype=np.int8)
y_dev = y_valid[dev_idx]
support = pd.DataFrame({"train":y_train.sum(0), "development":y_dev.sum(0),
    "calibration":y_valid[cal_idx].sum(0), "audit":y_valid[audit_idx].sum(0)}, index=LABELS)
display(support)
print("Validation row counts:", {"development":len(dev_idx), "calibration":len(cal_idx), "audit":len(audit_idx)})
if (support[["development","calibration","audit"]] < 10).any().any():
    warnings.warn("Some evaluation labels have fewer than 10 positives. Their F1 and thresholds are unstable; inspect per-label support.")

In [ ]:
# 6. Experiment identity: avoid silently reusing predictions from different data/settings.
PIPELINE_VERSION = "v3.1"
versions = {p: importlib.metadata.version(p) for p in ["torch","transformers","tokenizers","scikit-learn","numpy"]}
identity = {"pipeline":PIPELINE_VERSION, "labels":LABELS, "seed":SEED,
            "batch":TRAIN_BATCH_SIZE, "accumulation":ACCUMULATION_STEPS,
            "max_epochs":MAX_EPOCHS, "patience":PATIENCE, "length":MAX_LENGTH,
            "weight_decay":WEIGHT_DECAY, "warmup":WARMUP_RATIO, "versions":versions}
h = hashlib.sha256(json.dumps(identity, sort_keys=True).encode())
for df in [train_df, valid_df, test_df]:
    cols = [ID_COL,"clean_text","bert_text"] + (LABELS if set(LABELS).issubset(df.columns) else [])
    h.update(df[cols].to_csv(index=False).encode())
for idx in [dev_idx,cal_idx,audit_idx]:
    h.update(np.asarray(idx,dtype=np.int64).tobytes())
RUN_DIR = OUTPUT_ROOT / h.hexdigest()[:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
roles = np.full(len(valid_df), "audit", dtype=object)
roles[dev_idx], roles[cal_idx] = "development", "calibration"
pd.DataFrame({ID_COL:valid_df[ID_COL],"role":roles}).to_csv(RUN_DIR/"validation_split.csv",index=False)
print("Experiment folder:", RUN_DIR)

In [ ]:
# 7. Metrics and restrained threshold selection.
def macro_f1(y, probabilities, thresholds=.5):
    return float(f1_score(y, probabilities >= np.asarray(thresholds), average="macro", zero_division=0))

def tune_thresholds(y, probabilities):
    # Coarser than the old 0.01 grid. Ties prefer the threshold nearest 0.5.
    grid = np.round(np.arange(.10, .901, .025), 3)
    thresholds = np.full(y.shape[1], .5, dtype=float)
    for j in range(y.shape[1]):
        positives = int(y[:,j].sum())
        if min(positives, len(y)-positives) < 5:
            continue  # insufficient support: do not fit an arbitrary extreme cutoff
        values = np.array([f1_score(y[:,j], probabilities[:,j] >= t, zero_division=0) for t in grid])
        tied = np.flatnonzero(np.isclose(values, values.max(), atol=1e-10, rtol=0))
        chosen = grid[tied[np.argmin(abs(grid[tied]-.5))]]
        # Modest shrinkage for low-support labels reduces unstable extremes.
        strength = min(1.0, positives / 30.0)
        thresholds[j] = .5 + strength * (chosen-.5)
    return thresholds

def label_report(y, probabilities, thresholds):
    p,r,f,s = precision_recall_fscore_support(y, probabilities >= thresholds, zero_division=0)
    return pd.DataFrame({"label":LABELS,"positives":s,"precision":p,"recall":r,"f1":f,"threshold":thresholds})

In [ ]:
# 8. TF-IDF baseline. Choose C and class weighting on development only.
# Whitespace tokens preserve Bangla combining marks that sklearn's default word regex can split.
word = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
    ngram_range=(1,2), min_df=2, max_features=100_000, sublinear_tf=True, dtype=np.float32)
char = TfidfVectorizer(analyzer="char", lowercase=False, ngram_range=(2,5),
    min_df=2, max_features=150_000, sublinear_tf=True, dtype=np.float32)
X_train = hstack([word.fit_transform(train_df.clean_text), char.fit_transform(train_df.clean_text)],format="csr")
X_valid = hstack([word.transform(valid_df.clean_text), char.transform(valid_df.clean_text)],format="csr")
X_test = hstack([word.transform(test_df.clean_text), char.transform(test_df.clean_text)],format="csr")
tfidf_valid = np.zeros(y_valid.shape,dtype=np.float32)
tfidf_test = np.zeros((len(test_df),len(LABELS)),dtype=np.float32)
tfidf_settings = []
model = best = None
for j,label in enumerate(tqdm(LABELS,desc="TF-IDF labels")):
    if len(np.unique(y_train[:,j])) < 2:
        value = float(y_train[:,j].mean())
        tfidf_valid[:,j], tfidf_test[:,j] = value, value
        tfidf_settings.append({"label":label,"constant":value})
        continue
    best = None
    for weight in [None, "balanced"]:
        for c in [.5, 2., 8.]:
            model = LogisticRegression(C=c,class_weight=weight,solver="liblinear",max_iter=2000,random_state=SEED)
            model.fit(X_train,y_train[:,j])
            p = model.predict_proba(X_valid[dev_idx])[:,1,None]
            threshold = tune_thresholds(y_dev[:,[j]],p)
            score = macro_f1(y_dev[:,[j]],p,threshold)
            if best is None or score > best[0]+1e-8:
                best = (score,model,c,weight)
    score,model,c,weight = best
    tfidf_valid[:,j] = model.predict_proba(X_valid)[:,1]
    tfidf_test[:,j] = model.predict_proba(X_test)[:,1]
    tfidf_settings.append({"label":label,"C":c,"class_weight":weight,"development_f1_tuned":score})
np.savez_compressed(RUN_DIR/"tfidf_predictions.npz",valid=tfidf_valid,test=tfidf_test)
display(pd.DataFrame(tfidf_settings))
del X_train,X_valid,X_test,word,char,model,best
gc.collect()

## Transformer training
A **learning rate** controls how large each training update is. An **epoch** is one pass through the training examples. The best epoch is selected using development macro F1 at a fixed threshold of 0.5; thresholds are tuned later on separate calibration examples.

Standard mode compares learning rates 2e-5 and 1e-5. Thorough mode additionally tries a second seed, BanglishBERT, and mild positive-class weighting. These are candidates, not claims that one model is universally best. Completed runs can be reused from Drive. An interrupted individual run restarts from epoch 1; this notebook does not resume partial optimizer state.

In [ ]:
# 9. Tokenize once per run; dynamic padding; explicit float32 multilabel collation.
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, text_column, with_labels=False):
        self.tokens = tokenizer(df[text_column].tolist(), truncation=True, max_length=max_length, padding=False)
        self.labels = df[LABELS].to_numpy(dtype=np.float32) if with_labels else None
    def __len__(self):
        return len(self.tokens["input_ids"])
    def __getitem__(self,index):
        item = {k:v[index] for k,v in self.tokens.items()}
        if self.labels is not None:
            item["labels"] = self.labels[index]
        return item

class MultiLabelCollator:
    def __init__(self,tokenizer): self.tokenizer = tokenizer
    def __call__(self,items):
        features = [{k:v for k,v in item.items() if k!="labels"} for item in items]
        batch = self.tokenizer.pad(features,padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if "labels" in items[0]:
            batch["labels"] = torch.tensor(np.stack([item["labels"] for item in items]),dtype=torch.float32)
        return batch

def loader_for(df, tokenizer, cfg, training=False):
    column = "bert_text" if cfg["model"].startswith("csebuetnlp/") else "clean_text"
    ds = TextDataset(df,tokenizer,cfg["max_length"],column,with_labels=training)
    return DataLoader(ds,batch_size=TRAIN_BATCH_SIZE if training else EVAL_BATCH_SIZE,
        shuffle=training,num_workers=0,pin_memory=True,collate_fn=MultiLabelCollator(tokenizer))

@torch.no_grad()
def predict(model,loader):
    model.eval()
    chunks=[]
    for batch in loader:
        batch={k:v.to(DEVICE) for k,v in batch.items() if k!="labels"}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**batch).logits
        chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
    p=np.concatenate(chunks)
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    return p

In [ ]:
# 10. One GPU model at a time, disk checkpoints, and paired validation/test caches.
def train_run(cfg):
    key=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:10]
    folder=RUN_DIR/(cfg["name"]+"_"+key)
    folder.mkdir(exist_ok=True)
    cache,meta_path=folder/"predictions.npz",folder/"run.json"
    if REUSE_COMPLETED_RUNS and cache.exists() and meta_path.exists():
        meta=json.loads(meta_path.read_text())
        assert meta["config"]==cfg
        with np.load(cache,allow_pickle=False) as z:
            vp,tp=z["valid"].copy(),z["test"].copy()
        assert vp.shape==y_valid.shape and tp.shape==(len(test_df),len(LABELS))
        assert np.isfinite(vp).all() and np.isfinite(tp).all()
        print("Reusing completed run:",cfg["name"])
        return {"name":cfg["name"],"valid":vp,"test":tp,"meta":meta}
    seed_everything(cfg["seed"])
    tokenizer=AutoTokenizer.from_pretrained(cfg["model"])
    column="bert_text" if cfg["model"].startswith("csebuetnlp/") else "clean_text"
    probe=train_df[column].sample(min(2000,len(train_df)),random_state=SEED).tolist()
    lengths=np.array([len(x) for x in tokenizer(probe,truncation=False)["input_ids"]])
    print(cfg["name"],"| token length p50/p90/p95/p99:",np.percentile(lengths,[50,90,95,99]).round())
    print("Fraction exceeding max length:",round(float((lengths>cfg["max_length"]).mean()),4))
    train_loader=loader_for(train_df,tokenizer,cfg,True)
    dev_loader=loader_for(valid_df.iloc[dev_idx],tokenizer,cfg)
    model=AutoModelForSequenceClassification.from_pretrained(cfg["model"],num_labels=len(LABELS),
        problem_type="multi_label_classification",id2label=dict(enumerate(LABELS)),
        label2id={v:k for k,v in enumerate(LABELS)}).to(DEVICE)
    # Biases and normalization parameters are excluded from weight decay.
    groups=[{"params":[p for n,p in model.named_parameters() if p.requires_grad and p.ndim>1],"weight_decay":WEIGHT_DECAY},
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and p.ndim<=1],"weight_decay":0.0}]
    optimizer=torch.optim.AdamW(groups,lr=cfg["lr"])
    total_steps=math.ceil(len(train_loader)/ACCUMULATION_STEPS)*MAX_EPOCHS
    scheduler=get_linear_schedule_with_warmup(optimizer,int(total_steps*WARMUP_RATIO),total_steps)
    scaler=torch.cuda.amp.GradScaler()
    if cfg["pos_weight"]:
        pos=y_train.sum(0)
        weights=np.clip(np.sqrt((len(y_train)-pos)/np.maximum(pos,1)),1,5)
        loss_fn=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    else:
        loss_fn=torch.nn.BCEWithLogitsLoss()
    best_score,best_epoch,stale=-1.,0,0
    history=[]
    checkpoint=folder/"best_state.pt"
    for epoch in range(1,MAX_EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum=0.
        for step,batch in enumerate(tqdm(train_loader,desc=f"{cfg['name']} epoch {epoch}/{MAX_EPOCHS}")):
            target=batch.pop("labels").to(DEVICE)
            inputs={k:v.to(DEVICE) for k,v in batch.items()}
            # Scale by real sample count, including the final partial accumulation group.
            group_start=(step//ACCUMULATION_STEPS)*ACCUMULATION_STEPS
            group_samples=min(ACCUMULATION_STEPS*TRAIN_BATCH_SIZE,len(train_loader.dataset)-group_start*TRAIN_BATCH_SIZE)
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inputs).logits
                loss=loss_fn(logits,target)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss; inspect input labels and learning rate.")
            scaler.scale(loss*(target.shape[0]/group_samples)).backward()
            loss_sum+=loss.item()*target.shape[0]
            if (step+1)%ACCUMULATION_STEPS==0 or step+1==len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                old_scale=scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale()>=old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        p=predict(model,dev_loader)
        score=macro_f1(y_dev,p,.5)
        history.append({"epoch":epoch,"train_loss":loss_sum/len(train_df),"development_macro_f1_at_0_5":score})
        pd.DataFrame(history).to_csv(folder/"history.csv",index=False)
        print(history[-1])
        if score>best_score+1e-6:
            best_score,best_epoch,stale=score,epoch,0
            # CPU copy avoids retaining a second complete GPU model.
            state={k:v.detach().cpu() for k,v in model.state_dict().items()}
            temp=folder/"best_state.tmp"
            torch.save(state,temp)
            temp.replace(checkpoint)
            del state
        else:
            stale+=1
            if stale>=PATIENCE:
                break
    # Release optimizer GPU state before full inference.
    del optimizer,scheduler,scaler,groups,train_loader,dev_loader
    gc.collect(); torch.cuda.empty_cache()
    model.load_state_dict(torch.load(checkpoint,map_location="cpu",weights_only=True))
    model.save_pretrained(folder/"best_model",safe_serialization=True)
    tokenizer.save_pretrained(folder/"best_model")
    checkpoint.unlink(missing_ok=True)
    vp=predict(model,loader_for(valid_df,tokenizer,cfg))
    tp=predict(model,loader_for(test_df,tokenizer,cfg))
    meta={"config":cfg,"best_epoch":best_epoch,"development_macro_f1_at_0_5":best_score,
          "history":history,"resolved_model_commit":getattr(model.config,"_commit_hash",None)}
    temporary=folder/"predictions.tmp.npz"
    np.savez_compressed(temporary,valid=vp,test=tp)
    temporary.replace(cache)
    meta_path.write_text(json.dumps(meta,indent=2))  # marker written last: only completed runs reused
    del model,tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return {"name":cfg["name"],"valid":vp,"test":tp,"meta":meta}

In [ ]:
# 11. Run the planned experiments. Standard = two fits, not a large grid search.
def make_config(name,lr=2e-5,seed=42,model="csebuetnlp/banglabert",pos_weight=False):
    return dict(name=name,lr=lr,seed=seed,model=model,pos_weight=pos_weight,max_length=MAX_LENGTH)
configs=[make_config("banglabert_lr2e5_s42")]
if PRESET in {"standard","thorough"}:
    configs.append(make_config("banglabert_lr1e5_s42",lr=1e-5))
if PRESET=="thorough":
    configs.extend([make_config("banglabert_lr2e5_s123",seed=123),
                    make_config("banglishbert_lr2e5_s42",model="csebuetnlp/banglishbert"),
                    make_config("banglabert_weighted_lr2e5_s42",pos_weight=True)])
# Optional later experiment: append make_config("xlmr_lr2e5_s42", model="FacebookAI/xlm-roberta-base")
results=[]
for cfg in configs:
    results.append(train_run(cfg))
print("Completed transformer fits:",len(results))

In [ ]:
# 12. Actual probability ensembles, using the SAME run members for validation and test.
def mean_candidate(name,members):
    return {"name":name,"valid":np.mean([r["valid"] for r in members],axis=0),
            "test":np.mean([r["test"] for r in members],axis=0),
            "members":[r["name"] for r in members]}
transformer_candidates=[{**r,"members":[r["name"]]} for r in results]
if len(results)>1:
    transformer_candidates.append(mean_candidate("all_transformers_mean",results))
seed_members=[r for r in results if r["meta"]["config"]["model"]=="csebuetnlp/banglabert"
              and r["meta"]["config"]["lr"]==2e-5 and not r["meta"]["config"]["pos_weight"]]
if len(seed_members)>1:
    transformer_candidates.append(mean_candidate("banglabert_seed_mean",seed_members))
# Select a small set of global mixture weights. No per-label weight search on this small holdout.
rows=[]
best=None
for candidate in transformer_candidates:
    for weight in [0.,.25,.5,.75,1.]:
        vp=weight*tfidf_valid+(1-weight)*candidate["valid"]
        thresholds=tune_thresholds(y_valid[cal_idx],vp[cal_idx])
        score=macro_f1(y_valid[cal_idx],vp[cal_idx],thresholds)
        rows.append({"transformer":candidate["name"],"tfidf_weight":weight,"calibration_macro_f1":score})
        if best is None or score>best["calibration_macro_f1"]+1e-8:
            best={**rows[-1],"members":candidate["members"],"thresholds":thresholds,
                  "valid":vp,"test":weight*tfidf_test+(1-weight)*candidate["test"]}
leaderboard=pd.DataFrame(rows).sort_values("calibration_macro_f1",ascending=False)
leaderboard.to_csv(RUN_DIR/"experiment_leaderboard.csv",index=False)
display(leaderboard.head(10))
print("Selected transformer members:",best["members"],"| TF-IDF weight:",best["tfidf_weight"])
print("Calibration score is a selection score; it is optimistic. Audit is reported next.")

In [ ]:
# 13. One final audit. Freeze the model/weights/thresholds before examining these labels.
audit_y=y_valid[audit_idx]
audit_p=best["valid"][audit_idx]
audit_score=macro_f1(audit_y,audit_p,best["thresholds"])
audit_table=label_report(audit_y,audit_p,best["thresholds"])
audit_table.to_csv(RUN_DIR/"per_label_audit.csv",index=False)
print(f"Final audit macro F1: {audit_score:.4f}")
print("Your reported Kaggle baseline: 0.516 (different data; these numbers are not directly comparable).")
display(audit_table.round(4))
# Export errors for interpretation, not another round of tuning on this same audit.
errors=valid_df.iloc[audit_idx][[ID_COL,TEXT_COL]].copy()
for j,label in enumerate(LABELS):
    errors[label+"_true"]=audit_y[:,j]
    errors[label+"_prob"]=audit_p[:,j]
    errors[label+"_pred"]=(audit_p[:,j]>=best["thresholds"][j]).astype(int)
errors["wrong_labels"]=(audit_y!=(audit_p>=best["thresholds"])).sum(1)
errors.loc[errors.wrong_labels>0].sort_values("wrong_labels",ascending=False).to_csv(RUN_DIR/"audit_errors.csv",index=False)

In [ ]:
# 14. Build binary submission in EXACT sample-submission order.
def make_submission(test_ids, sample_frame, probabilities, thresholds):
    probabilities=np.asarray(probabilities)
    assert probabilities.shape==(len(test_ids),len(LABELS))
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    assert len(set(test_ids))==len(test_ids)
    assert sample_frame[ID_COL].is_unique and set(sample_frame[ID_COL])==set(test_ids)
    pred=(probabilities>=np.asarray(thresholds)).astype(np.int8)
    frame=pd.DataFrame(pred,columns=LABELS,index=pd.Index(test_ids,name=ID_COL))
    frame=frame.loc[sample_frame[ID_COL].tolist()].reset_index()[sample_frame.columns.tolist()]
    assert frame[ID_COL].tolist()==sample_frame[ID_COL].tolist()
    assert frame[LABELS].isin([0,1]).all().all() and not frame.isna().any().any()
    return frame

submission=make_submission(test_df[ID_COL].tolist(),sample,best["test"],best["thresholds"])
SUBMISSION_PATH=RUN_DIR/"submission_optimized_v3.csv"
submission.to_csv(SUBMISSION_PATH,index=False)
# Round-trip check catches accidental formatting / row alignment mistakes.
reloaded=pd.read_csv(SUBMISSION_PATH,dtype={ID_COL:str})
assert reloaded[ID_COL].tolist()==sample[ID_COL].tolist()
assert np.array_equal(reloaded[LABELS].to_numpy(),submission[LABELS].to_numpy())
print("READY TO SUBMIT:",SUBMISSION_PATH)
display(submission.head())
display(pd.DataFrame({"label":LABELS,"train_positive_rate":y_train.mean(0),
    "test_predicted_positive_rate":submission[LABELS].mean().values}))

In [ ]:
# 15. Save a small results bundle; no Kaggle credential or large model checkpoint goes into it.
report={"pipeline":PIPELINE_VERSION,"reported_previous_kaggle_score":.516,"new_kaggle_score":None,
    "metric_assumption":METRIC,"preset":PRESET,"identity":identity,
    "data_rows":{"train":len(train_df),"validation":len(valid_df),"test":len(test_df)},
    "split_rows":{"development":len(dev_idx),"calibration":len(cal_idx),"audit":len(audit_idx)},
    "tfidf_settings":tfidf_settings,"runs":[r["meta"] for r in results],
    "selection":{"transformer":best["transformer"],"members":best["members"],
        "tfidf_weight":best["tfidf_weight"],"calibration_macro_f1":best["calibration_macro_f1"],
        "audit_macro_f1":audit_score,"thresholds":dict(zip(LABELS,best["thresholds"].tolist()))},
    "note":"Audit is untouched during selection; no improvement claim until Kaggle evaluates the new CSV."}
(RUN_DIR/"experiment_report.json").write_text(json.dumps(report,indent=2))
np.savez_compressed(RUN_DIR/"selected_predictions.npz",valid=best["valid"],test=best["test"],
    thresholds=best["thresholds"],valid_ids=valid_df[ID_COL].to_numpy(dtype=str),
    test_ids=test_df[ID_COL].to_numpy(dtype=str),labels=np.asarray(LABELS))
BUNDLE_PATH=RUN_DIR/"v3_results.zip"
with zipfile.ZipFile(BUNDLE_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file() and p.suffix in {".csv",".json",".npz"}:
            z.write(p,p.name)
    for p in RUN_DIR.glob("*/predictions.npz"):
        z.write(p,str(p.relative_to(RUN_DIR)))
    for p in RUN_DIR.glob("*/run.json"):
        z.write(p,str(p.relative_to(RUN_DIR)))
print("Saved submission and results bundle:",SUBMISSION_PATH,BUNDLE_PATH,sep="\n")
from google.colab import files
files.download(str(SUBMISSION_PATH))
files.download(str(BUNDLE_PATH))

## After your first run
- Submit **submission_optimized_v3.csv** once. Record the new Kaggle score separately from the local audit score.
- Send back **v3_results.zip** plus the Kaggle score. It contains per-label results, training histories, selected settings, and reusable probabilities, so the next decision can be based on evidence.
- If GPU memory runs out, reduce training batch size to 4, raise accumulation to 4, and reduce evaluation batch size to 8. Restart the runtime to release memory, then rerun.
- If training truncates many examples, try `MAX_LENGTH = 256` as one planned follow-up. A length change retrains models and creates a different experiment folder.
- Use `thorough` only if you have the GPU time. Standard runs are reused when their data/settings match. Extra models may perform worse; selection can give TF-IDF or a transformer all the weight.
- Keep public leaderboard experiments few and deliberate. After repeated tuning, the audit is no longer independent evidence; use a new holdout or nested cross-validation for a more rigorous estimate.

## Deliberate limits
No hidden labels, synthetic training data, automatic label hierarchy, train/test duplicate label copying, or unverified external labeled corpus is used. Validation is not added to training after calibration: doing so changes prediction calibration and requires a separately evaluated refitting procedure. If validation is tiny, a future grouped out-of-fold pipeline can use data more efficiently, but costs more training runs.

The code was checked for Python syntax and its CPU-side splitting, thresholding, ensembling, and submission alignment were tested on synthetic inputs. The actual GPU training, competition data download, normalizer installation, and full Colab runtime have **not** been executed here. The input attachment contains code only, and this environment has no PyTorch/GPU training setup or competition CSVs.

## References
- [BanglaBERT model card and required preprocessing](https://huggingface.co/csebuetnlp/banglabert/blob/main/README.md): the authors specify their normalization pipeline before tokenization. This is why V3 adds model-specific normalization.
- [Official BUET normalizer](https://github.com/csebuetnlp/normalizer): dependency used for that preprocessing.
- [Hugging Face padding/collator documentation](https://huggingface.co/docs/transformers/v4.44.2/en/main_classes/data_collator): batch padding; V3 explicitly stacks multilabel targets as floats.
- [Competition page](https://www.kaggle.com/competitions/bengali-nlp-safety-challenge-2026): verify scoring and model-use rules there; not independently accessible during this edit.